# CUAD evaluation with vLLM (Kaggle)

Runs the baselines and the fine-tuned model on the CUAD test split and scores them.

**Notebook settings:** Accelerator **GPU T4 x2**, Internet **On**. For the fine-tuned tasks, **Add Input →
the training notebook's output** and set `TRAIN_OUTPUT` below.

**Tasks** (put the ones to run this session in `TASKS`; each prints its own scores):
| Task | What | Rough T4 time |
|---|---|---|
| `zeroshot_4b` | Qwen3-4B-Instruct, no fine-tuning | ~1 h |
| `zeroshot_8b` | Qwen3-8B, no fine-tuning (split over both T4s) | ~1.5 h |
| `rag_8b` | Hybrid retrieval + Qwen3-8B | ~2.5–3 h |
| `select_checkpoint` | Score the final adapter and the lowest-eval-loss checkpoint on **val**; pick the best | ~1–1.5 h |
| `test_finetuned` | Score the chosen adapter (`BEST`) on **test** | ~1 h |

Times are estimates. Set `LIMIT = 3` for a quick smoke test first. Responses are cached in
`outputs/<run>/responses.jsonl`: if a session dies, attach this notebook's output as input and copy
`outputs/` back to resume without redoing finished requests.

If Qwen3-8B produces garbage in fp16, swap `Qwen/Qwen3-8B` for `Qwen/Qwen3-8B-AWQ` in the serve call.

In [ ]:
REPO_URL = "https://github.com/lakshy-606/cuad-qlora-vllm.git"
TASKS = ["zeroshot_4b"]  # any of: zeroshot_4b, zeroshot_8b, rag_8b, select_checkpoint, test_finetuned
TRAIN_OUTPUT = "/kaggle/input/YOUR-TRAINING-NOTEBOOK/cuad-qlora-vllm/outputs/qwen3_4b_cuad_lora"
BEST = None  # adapter name picked by select_checkpoint, e.g. "adapter" or "checkpoint-1200"
LIMIT = None  # e.g. 3 contracts for a smoke test

In [ ]:
def sh(cmd):
    """Run a shell command, streaming its output; raise if it fails."""
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get("_exit_code", 0)
    if code:
        raise RuntimeError(f"command failed with exit code {code}: {cmd}")

sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
%cd /kaggle/working
sh(f"rm -rf cuad-qlora-vllm && git clone -q {REPO_URL}")
%cd /kaggle/working/cuad-qlora-vllm
sh("pip install -q vllm")
sh("pip install -q -e .")
if "rag_8b" in TASKS:
    sh("pip install -q rank-bm25 faiss-cpu sentence-transformers")
sh("python scripts/prepare_data.py")

In [ ]:
import json
from pathlib import Path

BASE = "Qwen/Qwen3-4B-Instruct-2507"
# T4 has no bf16; both GPUs share each model (tensor parallel) for memory and speed.
T4 = "--dtype half --tensor-parallel-size 2 --max-model-len 12288 --gpu-memory-utilization 0.90"


def serve(model, extra=""):
    sh('if [ -f outputs/vllm.pid ]; then kill "$(cat outputs/vllm.pid)" 2>/dev/null; sleep 15; fi')
    sh(f"scripts/start_vllm.sh {model} {T4} {extra}")


def predict(config, *args):
    limit = f" --limit {LIMIT}" if LIMIT else ""
    sh(f"python scripts/predict.py --config {config} {' '.join(args)}{limit}")


def lora_args(adapters):
    mods = " ".join(f"{name}={path}" for name, path in adapters.items())
    return f"--enable-lora --max-lora-rank 16 --max-loras 1 --lora-modules {mods}"


def candidates():
    """The final adapter plus the checkpoint with the lowest val loss, if it's still on disk."""
    out = Path(TRAIN_OUTPUT)
    found = {"adapter": out / "adapter"}
    summary = json.loads((out / "train_summary.json").read_text())
    if summary["eval_losses"]:
        best = min(summary["eval_losses"], key=lambda e: e["eval_loss"])
        ckpt = out / "checkpoints" / f"checkpoint-{best['step']}"
        if ckpt.exists():
            found[ckpt.name] = ckpt
    return found

In [ ]:
if "zeroshot_4b" in TASKS:
    serve(BASE)
    predict("configs/zeroshot_qwen3_4b.yaml")
    sh("python scripts/evaluate.py outputs/zeroshot_qwen3_4b")

In [ ]:
if "zeroshot_8b" in TASKS or "rag_8b" in TASKS:
    serve("Qwen/Qwen3-8B")
    if "zeroshot_8b" in TASKS:
        predict("configs/zeroshot_qwen3_8b.yaml")
        sh("python scripts/evaluate.py outputs/zeroshot_qwen3_8b")
    if "rag_8b" in TASKS:
        predict("configs/rag_qwen3_8b.yaml")
        sh("python scripts/evaluate.py outputs/rag_qwen3_8b")

In [ ]:
# Checkpoint selection uses the val split only; the test split is touched once, by test_finetuned.
if "select_checkpoint" in TASKS:
    adapters = candidates()
    print("candidates:", list(adapters))
    serve(BASE, lora_args(adapters))
    runs = []
    for name in adapters:
        run = f"ft_val_{name}"
        predict("configs/finetuned_qwen3_4b.yaml", "--split val", f"--model {name}", f"--run-name {run}")
        runs.append(f"outputs/{run}")
    sh(f"python scripts/evaluate.py {' '.join(runs)}")
    scores = {r: json.loads(Path(r, "metrics.json").read_text())["coverage_f1"] for r in runs}
    best_run = max(scores, key=scores.get)
    print(f"Best on val: {best_run} (coverage F1 {scores[best_run]:.3f}); set BEST to its adapter name")

In [ ]:
if "test_finetuned" in TASKS:
    assert BEST, "set BEST to the adapter name picked by select_checkpoint"
    serve(BASE, lora_args({BEST: candidates()[BEST]}))
    predict("configs/finetuned_qwen3_4b.yaml", f"--model {BEST}")
    sh("python scripts/evaluate.py outputs/finetuned_qwen3_4b")

In [ ]:
# Scoreboard for every test run finished in this session, next to the published ContractEval numbers.
done = [str(p.parent) for p in Path("outputs").glob("*/predictions.jsonl") if not p.parent.name.startswith("ft_val_")]
if done:
    sh(f"python scripts/evaluate.py {' '.join(sorted(done))}")
sh('if [ -f outputs/vllm.pid ]; then kill "$(cat outputs/vllm.pid)" 2>/dev/null; fi')
sh("rm -rf data/raw")